## Adding data from FOIA requests

The data in the Chicago Data Portal only includes employee reimbursements paid after Jan. 1 of the previous calendar year. I am in the process of submitting FOIA requests to get data from further back, so that my site can host both historical and current data in one place.

After importing necessary packages, I define the path to the data and join path with anything files ending in the extension .xlsx (each file shared with my had the reimbursements paid in one quarter of one requested year, so my first return of 5 years of transactions gave me 20 excel files).

In [2]:
import pandas as pd
import glob
import os

27968 2009-2013

27009 (26989) 2014-2018

21511 (21491) 2019-2023

Total 76448 – this is confirmed when I run the cells

<hr>

## Reading in the Data

In [3]:
path = '/Users/hgorledeenn/Desktop/Chicago-Reimbursements-site/foia_data/all'
all_files = glob.glob(os.path.join(path, "*.xlsx"))
columns_to_use = ['Dept Code', 'Pymt Date', 'Vendor Name', 'Vendor Number', 'Voucher (Batch) Number', 'Invoice Line Nbr', 'Invoice Line Amount', 'Invoice Number', 'Invoice Date', 'Invoice Description']

I use a for loop to read in each excel file and merge them into one large file.

In [4]:
df_list = []

for filename in all_files:
    df_small = pd.read_excel(filename, usecols=columns_to_use)
    df_list.append(df_small)
    df = pd.concat(df_list, ignore_index=True)

df.head()

,Dept Code,Pymt Date,Vendor Name,Vendor Number,Voucher (Batch) Number,Invoice Line Nbr,Invoice Line Amount,Invoice Number,Invoice Date,Invoice Description
0,1.0,2019-02-21,"COLLINS, ADAM",10230254.0,PV01180100144,1,255.64,00144,2018-11-14,Travel 11/9-11/11/18
1,1.0,2019-02-21,"COLLINS, ADAM",10230254.0,PV01180100144,2,139.16,00144,2018-11-14,Travel 11/9-11/11/18
2,1.0,2019-02-21,"NEWBERN, TIFFANY G",10202626.0,PV01180100151,1,221.82,00151,2018-12-06,Travel 4/23-4/27/2018
3,1.0,2019-02-21,"NEWBERN, TIFFANY G",10202626.0,PV01180100151,2,144.68,00151,2018-12-06,Travel 4/23-4/27/2018
4,1.0,2019-02-21,"NEWBERN, TIFFANY G",10202626.0,PV01180100151,1,221.82,00151A,2018-12-06,Travel 4/30-5/3/2018


In [5]:
df.shape

(76468, 10)

<hr>

## Dealing with "Sum:" rows
There are 20 rows (one each from the 2019-2023 sheets) that has a "Sum" value for the sheet. I want to filter these out

In [6]:
df[df['Invoice Line Nbr'].str.contains("Sum", na=False)]

,Dept Code,Pymt Date,Vendor Name,Vendor Number,Voucher (Batch) Number,Invoice Line Nbr,Invoice Line Amount,Invoice Number,Invoice Date,Invoice Description
1370,NaN,NaT,NaN,NaN,NaN,Sum:,2253318.78,NaN,NaT,NaN
4288,NaN,NaT,NaN,NaN,NaN,Sum:,1551973.87,NaN,NaT,NaN
5741,NaN,NaT,NaN,NaN,NaN,Sum:,2083774.28,NaN,NaT,NaN
8852,NaN,NaT,NaN,NaN,NaN,Sum:,1663907.70,NaN,NaT,NaN
9775,NaN,NaT,NaN,NaN,NaN,Sum:,1797295.62,NaN,NaT,NaN
12799,NaN,NaT,NaN,NaN,NaN,Sum:,1829536.18,NaN,NaT,NaN
13550,NaN,NaT,NaN,NaN,NaN,Sum:,3223735.28,NaN,NaT,NaN
21927,NaN,NaT,NaN,NaN,NaN,Sum:,1625564.58,NaN,NaT,NaN
29328,NaN,NaT,NaN,NaN,NaN,Sum:,672054.80,NaN,NaT,NaN
33118,NaN,NaT,NaN,NaN,NaN,Sum:,2168619.10,NaN,NaT,NaN


In [7]:
df = df[~df['Invoice Line Nbr'].str.contains("Sum:", na=False)]

<hr>

## Fixing column types

In [8]:
df.dtypes

Dept Code                        float64
Pymt Date                 datetime64[ns]
Vendor Name                       object
Vendor Number                     object
Voucher (Batch) Number            object
Invoice Line Nbr                  object
Invoice Line Amount              float64
Invoice Number                    object
Invoice Date              datetime64[ns]
Invoice Description               object
dtype: object

the Dept Code is only an intiger, so it shouldn't be a float. I'll make it an int here as opposed to a string, though that could work too

In [9]:
df['Dept Code'] = df['Dept Code'].astype('int64')

I also want to make sure all my dates are the same format

In [10]:
df['Pymt Date'] = pd.to_datetime(df['Pymt Date'], format='mixed')
df['Invoice Date'] = pd.to_datetime(df['Invoice Date'], format='mixed')

`Vendor Number` is listed as type object, but it has a decimal place after it. I found three instances in the dataset of the Vendor Number column having non-integers in it.

My analysis currently doesn't rely on Vendor Number so I think I will err on the side of leaving in more data and not remove them from the dataset. I could also set these three rows in particular to NA values, but I don't think that's necessary at this stage

In [11]:
df[df['Vendor Number'].str.contains('E', na=False)]

,Dept Code,Pymt Date,Vendor Name,Vendor Number,Voucher (Batch) Number,Invoice Line Nbr,Invoice Line Amount,Invoice Number,Invoice Date,Invoice Description
17087,38,2018-01-16,"WICKERT JIMENEZ, SHEROZ R","WICKERT JIMENEZ, SHEROZ R",PV38173800832,1,30.0,5157064070202429303,2017-12-11,"WO#145911,146026,146134,146135"
17088,38,2018-01-16,"WICKERT JIMENEZ, SHEROZ R","WICKERT JIMENEZ, SHEROZ R",PV38173800832,2,1.0,5157064070202429303,2017-12-11,"WO#145911,146026,146134,146135"
17097,38,2018-01-30,"FRANK, MICHAEL A","FRANK, MICHAEL A",PV38173800831,1,189.0,19140002183077,2017-12-23,WO#146606


I want invoice line nbr to be an integer

In [12]:
df['Invoice Line Nbr'] = df['Invoice Line Nbr'].astype('int64')

In [13]:
df.dtypes

Dept Code                          int64
Pymt Date                 datetime64[ns]
Vendor Name                       object
Vendor Number                     object
Voucher (Batch) Number            object
Invoice Line Nbr                   int64
Invoice Line Amount              float64
Invoice Number                    object
Invoice Date              datetime64[ns]
Invoice Description               object
dtype: object

<hr>

## Checking for NAs

I want to make sure all my data was loaded in properly and that I've addressed above any instances where data was missed (resulting in an NA value)

In [14]:
df.columns

Index(['Dept Code', 'Pymt Date', 'Vendor Name', 'Vendor Number',
       'Voucher (Batch) Number', 'Invoice Line Nbr', 'Invoice Line Amount',
       'Invoice Number', 'Invoice Date', 'Invoice Description'],
      dtype='object')

In [15]:
df[df['Dept Code'].isna()]

,Dept Code,Pymt Date,Vendor Name,Vendor Number,Voucher (Batch) Number,Invoice Line Nbr,Invoice Line Amount,Invoice Number,Invoice Date,Invoice Description


In [16]:
df[df['Pymt Date'].isna()]

,Dept Code,Pymt Date,Vendor Name,Vendor Number,Voucher (Batch) Number,Invoice Line Nbr,Invoice Line Amount,Invoice Number,Invoice Date,Invoice Description


In [17]:
df[df['Vendor Name'].isna()]

,Dept Code,Pymt Date,Vendor Name,Vendor Number,Voucher (Batch) Number,Invoice Line Nbr,Invoice Line Amount,Invoice Number,Invoice Date,Invoice Description


In [18]:
df[df['Vendor Number'].isna()]

,Dept Code,Pymt Date,Vendor Name,Vendor Number,Voucher (Batch) Number,Invoice Line Nbr,Invoice Line Amount,Invoice Number,Invoice Date,Invoice Description


In [19]:
df[df['Voucher (Batch) Number'].isna()]

,Dept Code,Pymt Date,Vendor Name,Vendor Number,Voucher (Batch) Number,Invoice Line Nbr,Invoice Line Amount,Invoice Number,Invoice Date,Invoice Description


In [20]:
df[df['Invoice Line Nbr'].isna()]

,Dept Code,Pymt Date,Vendor Name,Vendor Number,Voucher (Batch) Number,Invoice Line Nbr,Invoice Line Amount,Invoice Number,Invoice Date,Invoice Description


In [21]:
df[df['Invoice Line Amount'].isna()]

,Dept Code,Pymt Date,Vendor Name,Vendor Number,Voucher (Batch) Number,Invoice Line Nbr,Invoice Line Amount,Invoice Number,Invoice Date,Invoice Description


In [22]:
df[df['Invoice Number'].isna()]

,Dept Code,Pymt Date,Vendor Name,Vendor Number,Voucher (Batch) Number,Invoice Line Nbr,Invoice Line Amount,Invoice Number,Invoice Date,Invoice Description


In [23]:
df[df['Invoice Date'].isna()]

,Dept Code,Pymt Date,Vendor Name,Vendor Number,Voucher (Batch) Number,Invoice Line Nbr,Invoice Line Amount,Invoice Number,Invoice Date,Invoice Description


In [24]:
df[df['Invoice Description'].isna()]

,Dept Code,Pymt Date,Vendor Name,Vendor Number,Voucher (Batch) Number,Invoice Line Nbr,Invoice Line Amount,Invoice Number,Invoice Date,Invoice Description
209,39,2019-01-03,"BEVERLY, ERIC BERNARD",10197446.0,PV39183931075,1,430.00,FI11061831075,2018-11-06,NaN
210,39,2019-01-24,"WHITE, GREGORY",10029425.0,PV39183930664,1,455.00,FI11061830664,2018-11-06,NaN
1371,27,2011-07-13,ALDERMAN RICARDO MUNOZ,1040266,PV15111552329,1,221.38,MISC-7/6/2011,2011-07-06,NaN
1385,41,2011-07-28,AGATHA LOWE RN PHD,1042457,PV41114101583,1,81.00,11/ 01583,2011-06-30,NaN
1387,41,2011-07-28,"BENBOW, NANETTE D",1044225,PV41114101331,1,79.00,11/ 01331,2011-06-07,NaN
...,...,...,...,...,...,...,...,...,...,...
72003,57,2012-08-24,"BREEN, MATTHEW S",53549022,PV57125790495,1,2940.00,125790495,2012-04-12,NaN
72007,57,2012-09-20,"THIRY, PATRICK M",50722021,PV57115790682,1,4380.00,115790682,2011-03-16,NaN
72094,15,2012-08-02,DEBORAH L. GRAHAM,55378021,PV15121552729,1,63.04,GAS REIMBURSEMENTS 7/24/12,2012-07-24,NaN
72126,15,2012-08-24,DEBRA SILVERSTEIN,55134030,PV15121553008,1,861.31,VARIOUS REIMBURSEMENTS 8/10/12,2012-08-10,NaN


<hr>

## NAs in `Invoice Description`

The above search found that NA values are only present in the `invoice description` column. This is good news (it seems most of the data wrangling from excel files went well) but also bad news, because the invoice description column has almost 10,000 NAs (about 12% of the data). This column, while not absolutely crucial for my dashboard, is nonetheless helpful for providing some of the context behind the transactions.

I want to better understand why they are NAs (were they in the data and didn't get copied correctly? Or was there just not a description listed in the data I got at all?)

In [26]:
df_na_id = df[df['Invoice Description'].isna()]
df_na_id.shape

(9866, 10)

In [27]:
df_na_id['Pymt Date'].value_counts()

Pymt Date
2009-08-31    412
2009-11-16    324
2010-03-26    270
2012-05-01    243
2009-10-20    238
             ... 
2012-02-02      1
2012-01-19      1
2012-01-18      1
2012-02-22      1
2022-05-26      1
Name: count, Length: 468, dtype: int64

The most popular payment date with NAs was 8-18-2009. When I look at that [excel sheet](/foia_data/2009-2013/Payment_Distribution_Details%20Q3%202009.xlsx), I do see that there are 412 rows with payment dates that match and with no listed description.

I did the same check for the next 2 most payment dates and the same was true. I suspect that this trend would continue through the rest of the dataset and that, because the rest of the columns loaded correctly, the lack of data comes from the excel sheets themselves and not from my wrangling.

<hr>

## Working with Invoice Line Items

There are some differences in this data as compared to the modern data I'm getting from my scraper. Specifically, some of these charges are reported as separate invoice line items, whereas in my data from the portal I just see the total for each invoice (non-itemized).

The added detail here is helpful for a more detailed analysis, but I think I would rather work to make this data match the new data I will be adding it to. It also could lead to a mistake, like when I count total invoices per department, these multi-row, single-invoice groups would artificially inflate that number.

#### $0 Invoices?

There are 169 rows where the invoice line amount is $0. I'm not totally sure why that would be the case, but I will keep them for now. It seems like some of them are one of multiple invoice line numbers, so I'll wait to remove them until after I've grouped the same invoices together.

In [28]:
df[df['Invoice Line Amount'] == 0]

,Dept Code,Pymt Date,Vendor Name,Vendor Number,Voucher (Batch) Number,Invoice Line Nbr,Invoice Line Amount,Invoice Number,Invoice Date,Invoice Description
1139,84,2019-02-04,"OGANOVICH, ROBERT G",10128255.0,PV84188407713,5,0.0,A155270972,2018-12-23,SAFETY SHOE REIMBURSEMENT
12583,59,2022-03-24,"GRAY, BARRY JOHN",10588575.0,PV59215900586,5,0.0,1272022B,2021-12-29,"REIMBURSEMENT FOR RIT CLASS, 11-28-21 TO 12-03..."
13809,41,2017-11-02,"AYALA, SAUL C",1057150,PV41174101696,2,0.0,101696-A,2017-10-26,"Reimbursement for CD Outbreak,"
42918,84,2014-09-23,SOLIMAN KHUDEIRA,50062141,PV84148402467,3,0.0,SK 08/06/14,2014-08-16,"TRAVEL REIMBURSEMENT FOR TRAVEL TO CALGARY, CA..."
50479,41,2022-09-23,BLESILDA GUILLEN,1058224.0,PV41224101636,1,0.0,20221636,2022-09-15,PARKING EXPENSES
...,...,...,...,...,...,...,...,...,...,...
58134,41,2022-11-03,"GRIMES, ANNETTE",10127547.0,PV41224101635,6,0.0,202216357,2022-09-15,PARKING EXPENSES
58135,41,2022-11-03,"GRIMES, ANNETTE",10127547.0,PV41224101635,6,0.0,202216358,2022-09-15,PARKING EXPENSES
61284,27,2009-07-29,"DAVILA, ISOLDA",1047374,EF27092700816,1,0.0,00816,2009-07-28,NaN
61285,27,2009-08-06,DAVID STAFFORD,1027559,EF27092700895,1,0.0,00895,2009-08-04,NaN


Some of the rows of the dataset represent only parts of an invoice paid (represented by a value>1 for the `Invoice Line Nbr` column). I grouped by the `Voucher (Batch) Number`, `Pymt Date` and `Invoice Description` columns to create a new dataframe of the total invoice amount, then merged in the other columns from the original dataframe.

There were some invoices that represented multiple days but were under one `Voucher (Batch) Number`. Because of this, I merged the two dataframes together on that column, as well as `Pymt Date` and `Invoice Description`.

In [29]:
df_sums = df.groupby(['Voucher (Batch) Number', 'Pymt Date', 'Invoice Description'], as_index=False)['Invoice Line Amount'].sum()

df_merged = pd.merge(df_sums,df[['Voucher (Batch) Number', 'Pymt Date', 'Invoice Description', 'Dept Code', 'Vendor Name', 'Vendor Number']],on=['Voucher (Batch) Number', 'Pymt Date', 'Invoice Description'], how='left')

df_merged_dropped = df_merged.drop_duplicates(subset=['Voucher (Batch) Number', 'Invoice Line Amount', 'Pymt Date', 'Invoice Description'], keep='first')

df_merged_dropped.head()

,Voucher (Batch) Number,Pymt Date,Invoice Description,Invoice Line Amount,Dept Code,Vendor Name,Vendor Number
0,PV01100100520,2011-01-07,Reimbursement for Phone usage,34.96,1,PATRICK L CAREY,50835027
1,PV01100100523,2011-01-07,Travel Reimbursement,173.20,1,TAMARA MAYBERRY,50082523
2,PV01100100524,2011-01-07,Travel Reimbursement,133.00,1,GERALD ALDER,53893023
3,PV01100100529,2011-01-18,Travel Reimbursement,422.13,1,A JOSEPH DEAL,50085828
4,PV01100100534,2011-01-24,Reimbursement for Phone Usage,85.18,1,PATRICK L CAREY,50835027


In [30]:
df_merged_dropped['Voucher (Batch) Number'].value_counts()

Voucher (Batch) Number
PV50105050680    21
PV50105050679    17
PV50105060969    16
PV50115060087    15
PV50135010941    13
                 ..
PV50135030258     1
PV50135030259     1
PV50135030260     1
PV50135030292     1
PVTX160502848     1
Name: count, Length: 55821, dtype: int64

<hr>

### Adding Department Names

The current data only includes `Dept Code` as a number. After exchanging emails with the finance department, I found out that a key can be found in the Annual Appropriation Ordinance. I used the [2025 Ordinance](https://www.chicago.gov/content/dam/city/depts/obm/supp_info/2025Budget/2025_Ordinance_Book_webVersion.pdf) to create the below dictionary, then mapped it onto my dataset.

In [31]:
dept_names = {1: 'OFFICE OF THE MAYOR',
              3: 'OFFICE OF THE INSPECTOR GENERAL',
              5: 'OFFICE OF BUDGET & MANAGEMENT',
              6: 'DEPARTMENT OF TECHNOLOGY AND INNOVATION',
              15: 'CITY COUNCIL',
              21: 'DEPARTMENT OF HOUSING',
              22: 'DEPARTMENT OF ZONING AND LAND USE PLANNING',
              23: 'DEPARTMENT OF CULTURAL AFFAIRS AND SPECIAL EVENTS',
              24: "MAYOR'S OFFICE OF SPECIAL EVENTS",
              25: 'OFFICE OF CITY CLERK',
              27: 'DEPARTMENT OF FINANCE',
              28: "CITY TREASURER'S OFFICE",
              29: 'DEPARTMENT OF REVENUE',
              30: 'DEPARTMENT OF ADMINISTRATIVE HEARINGS',
              31: 'DEPARTMENT OF LAW',
              32: 'OFFICE OF COMPLIANCE',
              33: 'DEPARTMENT OF HUMAN RESOURCES',
              35: 'DEPARTMENT OF PROCUREMENT SERVICES',
              38: 'DEPARTMENT OF FLEET AND FACILITY MANAGEMENT',
              39: 'BOARD OF ELECTION COMMISSIONERS',
              40: 'DEPARTMENT OF FLEET MANAGEMENT',
              41: 'CHICAGO DEPARTMENT OF PUBLIC HEALTH',
              45: 'CHICAGO COMMISSION ON HUMAN RELATIONS',
              48: "MAYORS OFFICE FOR PEOPLE WITH DISABILITIES",
              50: 'DEPARTMENT OF FAMILY AND SUPPORT SERVICES',
              51: 'OFFICE OF PUBLIC SAFETY ADMINISTRATION',
              54: 'DEPARTMENT OF PLANNING AND DEVELOPMENT',
              55: 'CHICAGO POLICE BOARD',
              56: 'INDEPENDENT POLICE REVIEW AUTHORITY',
              57: 'CHICAGO POLICE DEPARTMENT',
              58: 'OFFICE OF EMERGENCY MANAGEMENT AND COMMUNICATIONS',
              59: 'CHICAGO FIRE DEPARTMENT',
              60: 'CIVILIAN OFFICE OF POLICE ACCOUNTABILITY',
              62: 'COMMUNITY COMMISSION FOR PUBLIC SAFETY AND ACCOUNTABILITY',
              67: 'DEPARTMENT OF BUILDINGS',
              70: 'DEPARTMENT OF BUSINESS AFFAIRS AND CONSUMER PROTECTION',
              72: 'DEPARTMENT OF ENVIRONMENT',
              73: 'CHICAGO ANIMAL CARE AND CONTROL',
              77: 'LICENSE APPEAL COMMISSION',
              78: 'BOARD OF ETHICS',
              81: 'DEPARTMENT OF STREETS AND SANITATION',
              84: 'CHICAGO DEPARTMENT OF TRANSPORTATION',
              85: 'CHICAGO DEPARTMENT OF AVIATION',
              88: 'DEPARTMENT OF WATER MANAGEMENT',
              91: 'CHICAGO PUBLIC LIBRARY',
              99: 'FINANCE GENERAL'}

In [32]:
df_dept_names = df_merged_dropped
df_dept_names['Dept Name'] = df_dept_names['Dept Code'].map(dept_names)
df_dept_names.head()

/var/folders/0c/3z452hlj1pzb2yr_79hcgh8r0000gn/T/ipykernel_54258/3989363339.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_dept_names['Dept Name'] = df_dept_names['Dept Code'].map(dept_names)


,Voucher (Batch) Number,Pymt Date,Invoice Description,Invoice Line Amount,Dept Code,Vendor Name,Vendor Number,Dept Name
0,PV01100100520,2011-01-07,Reimbursement for Phone usage,34.96,1,PATRICK L CAREY,50835027,OFFICE OF THE MAYOR
1,PV01100100523,2011-01-07,Travel Reimbursement,173.20,1,TAMARA MAYBERRY,50082523,OFFICE OF THE MAYOR
2,PV01100100524,2011-01-07,Travel Reimbursement,133.00,1,GERALD ALDER,53893023,OFFICE OF THE MAYOR
3,PV01100100529,2011-01-18,Travel Reimbursement,422.13,1,A JOSEPH DEAL,50085828,OFFICE OF THE MAYOR
4,PV01100100534,2011-01-24,Reimbursement for Phone Usage,85.18,1,PATRICK L CAREY,50835027,OFFICE OF THE MAYOR


And I'll just make sure everything worked OK (no NAs were coerced)

In [33]:
df_na_dept_names = df_dept_names[df_dept_names['Dept Name'].isna()]
df_na_dept_names['Dept Code'].value_counts()

Dept Code
4     82
52    58
47    42
53    13
71    11
76     6
8      1
0      1
75     1
87     1
Name: count, dtype: int64

In [34]:
df_na_dept_names['Year'] = df_na_dept_names['Pymt Date'].dt.year

/var/folders/0c/3z452hlj1pzb2yr_79hcgh8r0000gn/T/ipykernel_54258/2116931566.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_na_dept_names['Year'] = df_na_dept_names['Pymt Date'].dt.year


In [ ]:
df_na_dept_names['Dept Code'].value_counts()

Dept Code
4     82
52    58
47    42
53    13
71    11
76     6
8      1
0      1
75     1
87     1
Name: count, dtype: int64

#### Missing info for all of these:
(they appear in my dataset but I could not find what depts they correspond to through chicago ordinances as far back as 2008)

0<br>4<br>8<br>47<br>52<br>53<br>71<br>75<br>76<br>87

<hr>

### Renaming columns

what

In [36]:
## Renaming column names for stylistic reasons
df_clean = df_dept_names.rename(columns={'Voucher (Batch) Number':'VOUCHER NUMBER',
                             'Invoice Line Amount':'AMOUNT',
                             'Pymt Date':'PAYMENT DATE',
                             'Vendor Name':'VENDOR NAME',
                             'Invoice Description':'DESCRIPTION',
                             'Dept Name':'DEPARTMENT'})

In [37]:
df_clean.to_csv('clean_foia_data.csv', index=False)